# Chapter 12: Project - News Aggregator

In this chapter, we build a **news aggregator chatbot** -- a system that collects and curates news articles from various sources, providing users with up-to-date information on topics of interest.

The chatbot will:
- **Fetch** news from multiple sources via APIs
- **Categorize** articles into topics using NLP (topic modeling)
- **Summarize** articles for quick consumption (extractive and abstractive)
- **Present** results through a Flask-based web interface

We cover the entire development lifecycle: design, data collection, preprocessing, NLP pipelines, UI construction, evaluation, and deployment.

## Setup and Installation

Install all required packages before proceeding.

In [ ]:
# Install required packages
!pip install requests beautifulsoup4 nltk scikit-learn gensim transformers flask

In [ ]:
# Download NLTK resources
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# Core imports used throughout the notebook
import json
import string
import time
import requests

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.probability import FreqDist
from sklearn.feature_extraction.text import TfidfVectorizer

print("All imports successful.")

---
## 12.1 Project Introduction and Design

### 12.1.1 Project Overview

The goal is to create a news aggregator chatbot that can:

1. Fetch news articles from multiple sources
2. Categorize news articles into different topics
3. Summarize news articles for quick consumption
4. Provide users with the latest news on demand

We use Python with libraries such as Requests, BeautifulSoup, NLTK, scikit-learn, Gensim, and Hugging Face Transformers, and integrate with the NewsAPI to fetch articles.

### 12.1.2 Design Considerations

Key design decisions:

| Consideration | Description |
|---|---|
| **Data Sources** | Reliable news sources and APIs (e.g., NewsAPI) |
| **Categorization** | NLP techniques to classify articles into topics (politics, sports, technology, entertainment) |
| **Summarization** | Algorithms to condense articles into short summaries |
| **User Interface** | Intuitive web UI for requesting and viewing news |
| **Scalability** | Handle many requests; fetch from multiple sources simultaneously |

### 12.1.3 System Architecture

The system consists of five components:

1. **Frontend Interface** -- The user interacts through a web app
2. **News Fetcher** -- Fetches articles from various sources using APIs
3. **NLP Engine** -- Processes, categorizes, and summarizes articles
4. **Database** -- Stores fetched articles, user preferences, and history
5. **Backend Server** -- Handles requests, orchestrates processing, returns responses

### 12.1.4 Implementation Plan

1. Define Requirements
2. Set Up Project Structure
3. Fetch News Articles
4. Process and Categorize News
5. Summarize News Articles
6. Build the User Interface
7. Integrate Components
8. Deploy the Chatbot

### 12.1.5 Project Structure

```
news_aggregator_chatbot/
  data/
    news_sources.json        # List of news sources and API endpoints
    articles.json            # Fetched news articles
  models/
    nlp_model.h5             # Trained NLP model
    tokenizer.pickle         # Serialized tokenizer
  scripts/
    news_fetcher.py          # Fetching news articles
    nlp_engine.py            # Processing and categorizing articles
    summarizer.py            # Summarizing articles
    chatbot_interface.py     # Chatbot frontend interface
  app.py                     # Main Flask application
  requirements.txt           # Project dependencies
```

### 12.1.6 Defining Requirements

The chatbot should be able to:

1. Fetch the latest news articles from multiple sources
2. Categorize articles into topics (politics, sports, technology, entertainment)
3. Summarize articles to provide concise information
4. Allow users to request news updates and view articles through an intuitive interface
5. Store user preferences and interaction history to personalize the experience

---
## 12.2 Data Collection and Preprocessing

Data collection and preprocessing are foundational steps. The quality of collected data and how it is processed directly impact the chatbot's accuracy and reliability.

This section covers:
- Collecting news articles from multiple sources via the NewsAPI
- Preprocessing: text normalization, tokenization, stop word removal, lemmatization, and TF-IDF vectorization

### 12.2.1 Collecting Data

We use the [NewsAPI](https://newsapi.org/) to fetch articles. You need to sign up for a free API key.

#### `news_sources.json` Configuration

This file lists the news sources and their API endpoints (replace `your_newsapi_api_key` with your actual key).

In [ ]:
# Define news sources configuration
news_sources_config = {
    "sources": [
        {
            "name": "BBC News",
            "url": "https://newsapi.org/v2/top-headlines?sources=bbc-news&apiKey=your_newsapi_api_key"
        },
        {
            "name": "CNN",
            "url": "https://newsapi.org/v2/top-headlines?sources=cnn&apiKey=your_newsapi_api_key"
        },
        {
            "name": "TechCrunch",
            "url": "https://newsapi.org/v2/top-headlines?sources=techcrunch&apiKey=your_newsapi_api_key"
        },
        {
            "name": "The Verge",
            "url": "https://newsapi.org/v2/top-headlines?sources=the-verge&apiKey=your_newsapi_api_key"
        }
    ]
}

# Preview the configuration
print(json.dumps(news_sources_config, indent=2))

#### Fetching News Articles

The `fetch_news()` function iterates over configured sources, calls each API endpoint, and collects article metadata (title, description, content, URL, publication date).

In [ ]:
def fetch_news(sources):
    """Fetch news articles from all configured sources."""
    articles = []
    for source in sources:
        response = requests.get(source["url"])
        if response.status_code == 200:
            news_data = response.json()
            for article in news_data["articles"]:
                articles.append({
                    "source": source["name"],
                    "title": article["title"],
                    "description": article["description"],
                    "content": article["content"],
                    "url": article["url"],
                    "publishedAt": article["publishedAt"]
                })
        else:
            print(f"Failed to fetch news from {source['name']}")
    return articles

# NOTE: Uncomment below to fetch live data (requires a valid API key)
# articles = fetch_news(news_sources_config["sources"])
# print(f"Fetched {len(articles)} articles")

#### Sample Data for Demonstration

Since the NewsAPI requires a valid API key, we create sample articles to demonstrate the rest of the pipeline.

In [ ]:
# Sample articles for demonstration purposes
sample_articles = [
    {
        "source": "BBC News",
        "title": "AI Advances in Healthcare",
        "description": "New AI models are transforming medical diagnostics.",
        "content": (
            "Artificial intelligence is making significant strides in healthcare. "
            "New machine learning models can now detect diseases from medical images "
            "with accuracy rivaling human doctors. Researchers at leading universities "
            "have developed deep learning systems that analyze X-rays, MRIs, and CT scans "
            "to identify conditions such as cancer, heart disease, and neurological disorders. "
            "These AI tools are being integrated into clinical workflows, helping doctors "
            "make faster and more accurate diagnoses. The technology is expected to reduce "
            "diagnostic errors and improve patient outcomes significantly."
        ),
        "url": "https://example.com/ai-healthcare",
        "publishedAt": "2024-01-15T10:00:00Z"
    },
    {
        "source": "CNN",
        "title": "Global Climate Summit Reaches Agreement",
        "description": "World leaders agree on new climate targets.",
        "content": (
            "At the Global Climate Summit, world leaders have reached a historic agreement "
            "to reduce carbon emissions by 50 percent by 2035. The agreement includes "
            "commitments to transition to renewable energy sources, invest in green technology, "
            "and support developing nations in their climate adaptation efforts. Environmental "
            "groups have praised the deal as a significant step forward, although some critics "
            "argue that the targets do not go far enough to prevent catastrophic climate change. "
            "Implementation plans are expected to be finalized in the coming months."
        ),
        "url": "https://example.com/climate-summit",
        "publishedAt": "2024-01-14T08:30:00Z"
    },
    {
        "source": "TechCrunch",
        "title": "New Smartphone Chip Breaks Performance Records",
        "description": "The latest mobile processor sets new benchmarks.",
        "content": (
            "A new smartphone chip has been unveiled that breaks all existing performance "
            "benchmarks. The processor features a cutting-edge architecture with improved "
            "energy efficiency and raw computing power. Early tests show it outperforms "
            "its predecessor by 40 percent in multi-core tasks while consuming 20 percent "
            "less power. The chip integrates an advanced neural processing unit for on-device "
            "AI tasks such as real-time translation, photo enhancement, and voice recognition. "
            "Major smartphone manufacturers are expected to adopt the chip in flagship devices "
            "launching later this year."
        ),
        "url": "https://example.com/new-chip",
        "publishedAt": "2024-01-13T14:00:00Z"
    },
    {
        "source": "The Verge",
        "title": "Streaming Platform Launches Interactive Movie",
        "description": "Viewers can now choose the storyline in a new interactive film.",
        "content": (
            "A major streaming platform has launched its first fully interactive movie, "
            "allowing viewers to make choices that affect the storyline and ending. The film "
            "features multiple branching narratives with over 20 possible endings. Using the "
            "remote control or touchscreen, audiences decide what the main character does at "
            "key moments. Critics have praised the technical achievement but noted that the "
            "storytelling quality varies across different paths. The platform plans to release "
            "more interactive content throughout the year, including a detective series."
        ),
        "url": "https://example.com/interactive-movie",
        "publishedAt": "2024-01-12T16:45:00Z"
    },
    {
        "source": "BBC News",
        "title": "Stock Markets Rally on Economic Data",
        "description": "Positive economic indicators boost investor confidence.",
        "content": (
            "Global stock markets rallied today following the release of better-than-expected "
            "economic data. Employment figures showed strong job growth, while inflation "
            "rates continued to decline. Investors responded positively, pushing major indices "
            "to new highs. Analysts attribute the rally to growing confidence in a soft landing "
            "for the economy. Technology stocks led the gains, with several major companies "
            "reporting quarterly earnings that exceeded expectations. Bond yields fell as "
            "markets priced in potential interest rate cuts later in the year."
        ),
        "url": "https://example.com/stock-rally",
        "publishedAt": "2024-01-11T09:15:00Z"
    }
]

print(f"Loaded {len(sample_articles)} sample articles")
for a in sample_articles:
    print(f"  - [{a['source']}] {a['title']}")

### 12.2.2 Preprocessing Data

Preprocessing converts raw text into a clean format suitable for NLP. Our pipeline:

1. **Text Normalization** -- convert to lowercase, remove punctuation
2. **Tokenization** -- split text into individual words
3. **Stop Word Removal** -- remove common words ("the", "is", "and", etc.)
4. **Lemmatization** -- reduce words to their base form ("running" -> "run")
5. **Vectorization** -- convert text to numerical TF-IDF features

In [ ]:
# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Clean and preprocess text: lowercase, tokenize, remove stopwords and punctuation, lemmatize."""
    # Convert to lowercase
    text = text.lower()
    # Tokenize
    tokens = nltk.word_tokenize(text)
    # Remove punctuation and stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in string.punctuation and word not in stop_words]
    # Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

# Demonstrate on a single sentence
demo_text = "The researchers are running multiple experiments on different datasets."
print(f"Original:     {demo_text}")
print(f"Preprocessed: {preprocess_text(demo_text)}")

In [ ]:
# Preprocess all sample articles
preprocessed_articles = []
for article in sample_articles:
    content = article["content"] if article["content"] else article["description"]
    preprocessed_content = preprocess_text(content)
    preprocessed_articles.append({
        "source": article["source"],
        "title": article["title"],
        "content": preprocessed_content,
        "original_content": article["content"],
        "url": article["url"],
        "publishedAt": article["publishedAt"]
    })

# Show a preprocessed example
print(f"Title: {preprocessed_articles[0]['title']}")
print(f"\nOriginal content (first 200 chars):\n  {sample_articles[0]['content'][:200]}...")
print(f"\nPreprocessed content:\n  {preprocessed_articles[0]['content']}")

#### TF-IDF Vectorization

TF-IDF (Term Frequency-Inverse Document Frequency) converts preprocessed text into numerical vectors. Words that are frequent in a document but rare across the corpus get higher weights.

In [ ]:
# Vectorize the preprocessed content using TF-IDF
vectorizer = TfidfVectorizer()
contents = [article["content"] for article in preprocessed_articles]
X = vectorizer.fit_transform(contents)

print(f"TF-IDF matrix shape: {X.shape}")
print(f"  {X.shape[0]} documents, {X.shape[1]} unique terms")
print(f"\nTop 20 terms in vocabulary:")
print(f"  {list(vectorizer.vocabulary_.keys())[:20]}")

In [ ]:
# In a production project, you would save the vectorizer and vectorized data to disk.
# For this notebook, we keep them in memory.
print("Vectorizer and TF-IDF matrix ready for downstream tasks.")

---
## 12.3 Implementing Text Summarization and Topic Modeling

With clean, vectorized data in hand, we now implement two core NLP capabilities:

1. **Text Summarization** -- Generate concise summaries of articles
2. **Topic Modeling** -- Automatically discover and assign topics to articles

### 12.3.1 Text Summarization

Two main approaches:

- **Extractive Summarization** -- Selects the most important sentences directly from the original text.
- **Abstractive Summarization** -- Generates new text that captures the main ideas (requires a language model).

#### Extractive Summarization with NLTK

We score each sentence by summing the frequencies of its non-stopword tokens, then select the top-scoring sentences.

In [ ]:
def summarize_text(text, num_sentences=3):
    """Extractive summarization: select top sentences by word frequency scoring."""
    # Tokenize into sentences
    sentences = sent_tokenize(text)

    # Tokenize into words, removing punctuation and stop words
    words = word_tokenize(text.lower())
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in string.punctuation and word not in stop_words]

    # Calculate word frequencies
    freq_dist = FreqDist(words)

    # Score sentences based on word frequencies
    sentence_scores = {}
    for sentence in sentences:
        for word in word_tokenize(sentence.lower()):
            if word in freq_dist:
                if sentence not in sentence_scores:
                    sentence_scores[sentence] = freq_dist[word]
                else:
                    sentence_scores[sentence] += freq_dist[word]

    # Select top N sentences
    summarized_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:num_sentences]

    # Combine into summary
    summary = ' '.join(summarized_sentences)
    return summary

In [ ]:
# Test extractive summarization on a sample NLP text
test_text = """
Natural language processing (NLP) is a subfield of linguistics, computer science, and
artificial intelligence concerned with the interactions between computers and human
language. In particular, it is about how to program computers to process and analyze
large amounts of natural language data. The result is a computer capable of
"understanding" the contents of documents, including the contextual nuances of the
language within them. The technology can then accurately extract information and
insights contained in the documents as well as categorize and organize the documents
themselves.
"""

summary = summarize_text(test_text, num_sentences=2)
print("Extractive Summary:")
print(summary)

In [ ]:
# Summarize each sample article
print("=" * 80)
print("EXTRACTIVE SUMMARIES OF ALL ARTICLES")
print("=" * 80)

for article in sample_articles:
    summary = summarize_text(article["content"], num_sentences=2)
    print(f"\n--- {article['title']} [{article['source']}] ---")
    print(f"Summary: {summary}")

#### Abstractive Summarization with Hugging Face Transformers

Abstractive summarization uses a pre-trained transformer model to generate new summary text that may not appear verbatim in the source.

> **Note:** This requires downloading a large model (~1.2 GB for the default `sshleifer/distilbart-cnn-12-6`). The cell is provided for reference; uncomment to run.

In [ ]:
# Abstractive Summarization using Hugging Face Transformers
# Uncomment to run (requires model download)

# from transformers import pipeline
#
# # Initialize the summarization pipeline
# summarizer_pipeline = pipeline("summarization")
#
# def abstractive_summarize_text(text, max_length=130, min_length=30):
#     """Generate an abstractive summary using a pre-trained transformer model."""
#     summary = summarizer_pipeline(text, max_length=max_length, min_length=min_length, do_sample=False)
#     return summary[0]['summary_text']
#
# # Test the abstractive summarizer
# abstract_summary = abstractive_summarize_text(test_text)
# print(f"Abstractive Summary: {abstract_summary}")

### 12.3.2 Topic Modeling with LDA (Gensim)

Topic modeling discovers abstract "topics" in a collection of documents. We use **Latent Dirichlet Allocation (LDA)** via the Gensim library.

Steps:
1. Tokenize the preprocessed content
2. Build a Gensim dictionary and bag-of-words corpus
3. Train an LDA model
4. Inspect discovered topics

In [ ]:
import gensim
from gensim import corpora

# Tokenize the preprocessed content
tokenized_contents = [nltk.word_tokenize(article["content"]) for article in preprocessed_articles]

# Create a Gensim dictionary
dictionary = corpora.Dictionary(tokenized_contents)

# Note: filter_extremes is useful for large corpora. With only 5 sample docs,
# we skip aggressive filtering to retain enough terms.
# dictionary.filter_extremes(no_below=5, no_above=0.5)

# Create bag-of-words corpus
corpus = [dictionary.doc2bow(text) for text in tokenized_contents]

print(f"Dictionary size: {len(dictionary)} terms")
print(f"Corpus size: {len(corpus)} documents")

In [ ]:
# Train the LDA model
num_topics = 3  # Using 3 topics for our small sample corpus
lda_model = gensim.models.ldamodel.LdaModel(
    corpus,
    num_topics=num_topics,
    id2word=dictionary,
    passes=15,
    random_state=42
)

# Print discovered topics
print("Discovered Topics:")
print("=" * 60)
topics = lda_model.print_topics(num_words=5)
for idx, topic in topics:
    print(f"  Topic {idx}: {topic}")

In [ ]:
# Categorize each article using the trained LDA model
def categorize_article(article_text, lda_model, dictionary):
    """Categorize an article by finding its dominant LDA topic."""
    tokens = nltk.word_tokenize(article_text.lower())
    bow = dictionary.doc2bow(tokens)
    topics = lda_model.get_document_topics(bow)
    sorted_topics = sorted(topics, key=lambda x: x[1], reverse=True)
    topic_index = sorted_topics[0][0]
    topic_terms = lda_model.show_topic(topic_index, topn=4)
    topic_keywords = [term[0] for term in topic_terms]
    return topic_index, topic_keywords

# Categorize all articles
print("Article Categorizations:")
print("=" * 60)
for article in preprocessed_articles:
    topic_idx, keywords = categorize_article(article["content"], lda_model, dictionary)
    print(f"  [{article['source']}] {article['title']}")
    print(f"    -> Topic {topic_idx}: {', '.join(keywords)}")

In [ ]:
# In production, save the LDA model and dictionary to disk:
# lda_model.save('models/lda_model')
# dictionary.save('models/dictionary.gensim')

print("LDA model and dictionary ready (save to disk for production use).")

---
## 12.4 Building the User Interface

We build a web-based UI using **Flask** (a lightweight Python web framework) and **Bootstrap** for responsive styling.

The UI enables users to:
- Select a news category
- Fetch and view articles
- Summarize and categorize individual articles

### 12.4.1 Setting Up Flask

The main Flask application defines API routes for fetching news, summarizing articles, and categorizing them.

#### `app.py`

In [ ]:
# Flask application code (for reference -- run as a standalone .py file, not in Jupyter)

flask_app_code = '''
from flask import Flask, request, jsonify, render_template
from nlp_engine import preprocess_text, categorize_article
from summarizer import summarize_text, abstractive_summarize_text
from news_fetcher import fetch_news

app = Flask(__name__)

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/get_news', methods=['POST'])
def get_news():
    category = request.form.get('category')
    articles = fetch_news(category)
    return jsonify(articles)

@app.route('/summarize', methods=['POST'])
def summarize():
    article = request.form.get('article')
    summary_type = request.form.get('summary_type')
    if summary_type == 'extractive':
        summary = summarize_text(article)
    else:
        summary = abstractive_summarize_text(article)
    return jsonify({'summary': summary})

@app.route('/categorize_article', methods=['POST'])
def categorize_article_route():
    article = request.form.get('article')
    category = categorize_article(article)
    return jsonify({'category': category})

@app.route('/summarize_article', methods=['POST'])
def summarize_article():
    article = request.form.get('article')
    summary_type = request.form.get('summary_type')
    if summary_type == 'extractive':
        summary = summarize_text(article)
    else:
        summary = abstractive_summarize_text(article)
    return jsonify({'summary': summary})

if __name__ == '__main__':
    app.run(debug=True)
'''

print(flask_app_code)

### 12.4.2 Creating HTML Templates

The HTML template uses Bootstrap for styling and jQuery for dynamic interactions. Users select a category, fetch articles, and can summarize or categorize each one.

#### `templates/index.html`

In [ ]:
# HTML template code (save as templates/index.html in the project)

html_template = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>News Aggregator Chatbot</title>
    <link rel="stylesheet"
          href="https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css">
</head>
<body>
    <div class="container">
        <h1 class="mt-5">News Aggregator Chatbot</h1>
        <form id="news-form" class="mt-4">
            <div class="form-group">
                <label for="category">Select News Category</label>
                <select class="form-control" id="category" name="category">
                    <option value="general">General</option>
                    <option value="business">Business</option>
                    <option value="technology">Technology</option>
                    <option value="sports">Sports</option>
                    <option value="entertainment">Entertainment</option>
                </select>
            </div>
            <button type="submit" class="btn btn-primary">Get News</button>
        </form>
        <div id="news-articles" class="mt-4"></div>
    </div>

    <script src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
    <script>
    $(document).ready(function() {
        $("#news-form").on("submit", function(event) {
            event.preventDefault();
            var category = $("#category").val();
            $.post("/get_news", { category: category }, function(data) {
                var articlesHtml = '<div class="list-group">';
                data.forEach(function(article) {
                    articlesHtml += '<div class="list-group-item">';
                    articlesHtml += '<h5 class="mb-1">' + article.title + '</h5>';
                    articlesHtml += '<p class="mb-1">' + article.description + '</p>';
                    articlesHtml += '<small>' + article.source + ' - ' + article.publishedAt + '</small>';
                    articlesHtml += '<br><button class="btn btn-secondary btn-sm mt-2 btn-categorize">Categorize</button>';
                    articlesHtml += ' <button class="btn btn-info btn-sm mt-2 btn-summarize">Summarize</button>';
                    articlesHtml += '<div class="category mt-2"></div>';
                    articlesHtml += '<div class="summary mt-2"></div>';
                    articlesHtml += '</div>';
                });
                articlesHtml += '</div>';
                $("#news-articles").html(articlesHtml);
            });
        });
    });
    </script>
</body>
</html>'''

print(html_template)

### 12.4.3 Integrating Summarization and Categorization

The Flask backend exposes endpoints:

| Endpoint | Method | Description |
|---|---|---|
| `/` | GET | Render the home page |
| `/get_news` | POST | Fetch articles by category |
| `/summarize_article` | POST | Summarize an article (extractive or abstractive) |
| `/categorize_article` | POST | Categorize an article using LDA |
| `/feedback` | POST | Collect user ratings for summaries |

The frontend uses jQuery AJAX calls to interact with these endpoints and dynamically update the page with results.

### 12.4.4 User Feedback Collection

To improve summarization quality over time, the application collects user feedback through a rating form (1-5 scale).

In [ ]:
# Flask feedback endpoint (add to app.py)

feedback_endpoint_code = '''
feedback_data = []

@app.route('/feedback', methods=['POST'])
def feedback():
    user_feedback = request.json
    feedback_data.append(user_feedback)
    return jsonify({'message': 'Thank you for your feedback!'})
'''

print(feedback_endpoint_code)
print("Users can rate summaries on a 1-5 scale via the web interface.")
print("Feedback is stored in memory and can be persisted to a database for analysis.")

---
## 12.5 Evaluating and Deploying the Aggregator

Evaluation ensures the chatbot meets quality standards. Deployment makes it accessible to users.

### 12.5.1 Evaluating the Aggregator

Three dimensions of evaluation:

1. **Accuracy Metrics** -- Precision, recall, F1-score for categorization
2. **User Feedback** -- Qualitative ratings for summarization quality
3. **Response Time** -- Latency for key operations

#### Evaluating Categorization Accuracy

In [ ]:
from sklearn.metrics import classification_report

# Simulated evaluation: compare predicted vs. true categories
# In production, you would use labeled test data

# Example true and predicted labels
true_categories = ['technology', 'politics', 'technology', 'entertainment', 'business']
predicted_categories = ['technology', 'politics', 'technology', 'entertainment', 'technology']

print("Classification Report:")
print("=" * 60)
print(classification_report(true_categories, predicted_categories, zero_division=0))

In [ ]:
# In a full project, evaluation would load test articles with ground-truth labels:

evaluate_code = '''
from sklearn.metrics import classification_report
from nlp_engine import categorize_article
import json

# Load test articles with ground-truth categories
with open('data/test_articles.json', 'r') as file:
    test_articles = json.load(file)

true_categories = [article['category'] for article in test_articles]
predicted_categories = [categorize_article(article['content']) for article in test_articles]

# Flatten for multi-label classification
true_flat = [item for sublist in true_categories for item in sublist]
pred_flat = [item for sublist in predicted_categories for item in sublist]

print(classification_report(true_flat, pred_flat))
'''

print(evaluate_code)

#### Measuring Response Time

Response time is critical for user experience. We measure how long key operations take.

In [ ]:
def measure_response_time(func, *args, **kwargs):
    """Measure the execution time of a function."""
    start_time = time.time()
    result = func(*args, **kwargs)
    end_time = time.time()
    response_time = end_time - start_time
    return result, response_time

# Measure summarization time
test_content = sample_articles[0]["content"]
summary, duration = measure_response_time(summarize_text, test_content, 2)
print(f"Extractive Summarization:")
print(f"  Time: {duration:.4f} seconds")
print(f"  Summary: {summary[:100]}...")

# Measure preprocessing time
_, duration = measure_response_time(preprocess_text, test_content)
print(f"\nPreprocessing:")
print(f"  Time: {duration:.4f} seconds")

# Measure categorization time
preprocessed = preprocess_text(test_content)
_, duration = measure_response_time(categorize_article, preprocessed, lda_model, dictionary)
print(f"\nLDA Categorization:")
print(f"  Time: {duration:.4f} seconds")

In [ ]:
# Measuring HTTP endpoint response time (when Flask app is running)

response_time_code = '''
import time
import requests

def measure_endpoint_response_time(endpoint, data):
    start_time = time.time()
    response = requests.post(endpoint, json=data)
    end_time = time.time()
    return end_time - start_time

article = "Sample news article content..."
data = {"article": article, "summary_type": "extractive"}
response_time = measure_endpoint_response_time(
    "http://localhost:5000/summarize_article", data
)
print(f"Summarization Response Time: {response_time} seconds")
'''

print(response_time_code)

### 12.5.2 Deploying the Aggregator

Once evaluated, the chatbot can be deployed as:
1. A **web application** on cloud platforms (Heroku, AWS, Google Cloud)
2. A **messaging bot** integrated with platforms like Facebook Messenger, Slack, or WhatsApp

#### Deploying on Heroku

Steps:
1. Install the Heroku CLI
2. Log in: `heroku login`
3. Create an app: `heroku create your-app-name`
4. Add a `Procfile` and `requirements.txt`
5. Push to Heroku: `git push heroku master`
6. Open: `heroku open`

In [ ]:
# Procfile content (save as 'Procfile' in the project root)
procfile = "web: python app.py"
print(f"Procfile:\n  {procfile}")

# requirements.txt content
requirements = """Flask
requests
nltk
gensim
transformers
scikit-learn"""

print(f"\nrequirements.txt:\n{requirements}")

# Deployment commands
print("\nDeployment commands:")
print("  heroku login")
print("  heroku create your-app-name")
print("  git init")
print("  git add .")
print('  git commit -m "Initial commit"')
print("  git push heroku master")
print("  heroku open")

#### Integrating with Facebook Messenger

The chatbot can also be deployed as a Messenger bot using Facebook's webhook API. The Flask app receives incoming messages and responds with summaries.

In [ ]:
# Facebook Messenger webhook integration (reference code)

messenger_code = '''
from flask import Flask, request
import requests

app = Flask(__name__)
PAGE_ACCESS_TOKEN = 'your_facebook_page_access_token'

def send_message(recipient_id, message_text):
    url = f"https://graph.facebook.com/v11.0/me/messages?access_token={PAGE_ACCESS_TOKEN}"
    headers = {"Content-Type": "application/json"}
    payload = {
        "recipient": {"id": recipient_id},
        "message": {"text": message_text}
    }
    requests.post(url, headers=headers, json=payload)

@app.route('/webhook', methods=['GET'])
def webhook_verification():
    mode = request.args.get('hub.mode')
    token = request.args.get('hub.verify_token')
    challenge = request.args.get('hub.challenge')
    if mode == 'subscribe' and token == 'your_verify_token':
        return challenge
    return 'Verification failed', 403

@app.route('/webhook', methods=['POST'])
def webhook():
    data = request.json
    if data['object'] == 'page':
        for entry in data['entry']:
            for messaging_event in entry['messaging']:
                sender_id = messaging_event['sender']['id']
                if 'message' in messaging_event:
                    message_text = messaging_event['message']['text']
                    response_text = summarize_text(message_text)
                    send_message(sender_id, response_text)
    return 'OK', 200

if __name__ == '__main__':
    app.run(debug=True)
'''

print(messenger_code)

---
## Chapter Summary

In this chapter, we built a complete **news aggregator chatbot** covering the full development lifecycle:

| Section | Key Activities |
|---|---|
| **12.1 Design** | Defined requirements, system architecture, and project structure |
| **12.2 Data Collection & Preprocessing** | Used NewsAPI for data collection; built a preprocessing pipeline (normalization, tokenization, stop word removal, lemmatization, TF-IDF vectorization) |
| **12.3 Summarization & Topic Modeling** | Implemented extractive summarization (NLTK) and abstractive summarization (Hugging Face Transformers); built LDA topic modeling (Gensim) |
| **12.4 User Interface** | Created a Flask web app with Bootstrap frontend; integrated summarization and categorization endpoints |
| **12.5 Evaluation & Deployment** | Measured accuracy (precision/recall/F1), response time, and user feedback; covered Heroku deployment and Facebook Messenger integration |

The skills and techniques learned here -- API integration, text preprocessing, summarization, topic modeling, web development, and deployment -- form a strong foundation for building more sophisticated NLP applications.